# PEFT Across Cognitive Regimes: Colab Pro Experiment Runner
**Methods:** `zero_shot`, `qlora`, `lora`, `dora`, `full_ft`  
**Tasks:** `esg` (Financial/ESG Multi-Class) & `folio` (Deductive Logical Reasoning)

In [ ]:
# 1. Mount Google Drive & Check GPU
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
# 2. Navigate to Project Folder & Install Dependencies
import os
PROJECT_DIR = '/content/drive/MyDrive/PEFT_LLM_Comparison_Study/peft_study_starter'
os.makedirs(PROJECT_DIR, exist_ok=True)
%cd $PROJECT_DIR
!pip install -q -r requirements.txt

In [ ]:
# 3. Sanity Check (100 samples, ~2 minutes) to Verify Pipeline & Drive Logging
!python src/run_experiment.py --model_key primary --task esg --method qlora --seed 42 --max_samples 100 --output_dir results/sanity

In [ ]:
# 4. Phase 1 Pilot Run (Seed=42) across All Methods on ESG & FOLIO
methods = ['zero_shot', 'qlora', 'lora', 'dora', 'full_ft']
tasks = ['esg', 'folio']

for task in tasks:
    for method in methods:
        res_path = f'results/res_primary_{task}_{method}_seed42.json'
        if os.path.exists(res_path):
            print(f'Skipping already completed: {res_path}')
            continue
        print(f'=== Running Task={task} | Method={method} | Seed=42 ===')
        !python src/run_experiment.py --model_key primary --task {task} --method {method} --seed 42 --output_dir results

In [ ]:
# 5. Aggregate Results into Summary CSV Table for Manuscript
import glob, json, pandas as pd
records = []
for fp in sorted(glob.glob('results/res_*.json')):
    with open(fp) as f:
        d = json.load(f)
    records.append({k: v for k, v in d.items() if k not in ['classification_report', 'confusion_matrix']})
df = pd.DataFrame(records)
df.to_csv('results/aggregated_results.csv', index=False)
display(df)